In [6]:
pip install numba numba-cuda

In [4]:
# vector of 1,000,000 elements and 256 threads per block => 3907 blocks, 8 warps per block
# thread 5 of block 10 => element 10*256 + 5 = 2565

import numpy as np
from numba import cuda

@cuda.jit
def kernel(x, block_idxs, thread_idxs):
    i = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x
    if i < block_idxs.size:
        block_idxs[i] = cuda.blockIdx.x
        thread_idxs[i] = cuda.threadIdx.x

n = 1_000_000
x = np.arange(n, dtype=np.float32)

# Copy input to the GPU and allocate the output there
d_x = cuda.to_device(x)
d_block_idxs = cuda.device_array_like(x)
d_thread_idxs = cuda.device_array_like(x)

threads_per_block = 256
blocks = (n + threads_per_block - 1) // threads_per_block
kernel[blocks, threads_per_block](d_x, d_block_idxs, d_thread_idxs)   # launch
block_idxs = d_block_idxs.copy_to_host()
thread_idxs = d_thread_idxs.copy_to_host()

print(f"Block: {block_idxs[2565]}")
print(f"Thread: {thread_idxs[2565]}")

Block: 10.0
Thread: 5.0


In [4]:
# coalescing exercise

import numpy as np
from numba import cuda

@cuda.jit
def copy_kernel(x, out):
    i = cuda.grid(1)
    if i < out.size:
        out[i] = x[i]

@cuda.jit
def copy_strided_kernel(x, out):
    i = cuda.grid(1)
    if i < out.size:
        out[i] = x[i*32]

n = 1_000_000
x = cuda.to_device(np.random.rand(n * 32).astype(np.float32))
out = cuda.device_array_like(x)

threads = 256
blocks = (n + threads - 1) // threads

def time_kernel(kernel):
  # warm-up
  kernel[blocks, threads](x, out)
  cuda.synchronize()

  start = cuda.event(timing=True)
  end = cuda.event(timing=True)

  n_iter = 100
  start.record()
  for _ in range(n_iter):
      kernel[blocks, threads](x, out)
  end.record()
  end.synchronize()

  total_ms = cuda.event_elapsed_time(start, end)
  return total_ms / n_iter

unstrided_time = time_kernel(copy_kernel)
strided_time = time_kernel(copy_strided_kernel)

print(f"avg copy kernel time (unstrided): {unstrided_time} ms")
print(f"avg copy kernel time (strided): {strided_time} ms")

avg copy kernel time (unstrided): 0.05001215934753418 ms
avg copy kernel time (strided): 0.2961011123657227 ms


In [8]:
# Hillis-Steele scan

import numpy as np
import numba
from numba import cuda

n = 8
x = cuda.to_device(np.arange(n).astype(np.float32))
out = cuda.device_array_like(x)

threads = n
blocks = 1

@cuda.jit
def scan_kernel(x, out):
  i = cuda.grid(1)
  if i < out.size:
    # shared buffer, initialize with x
    buf = cuda.shared.array(shape=(2, n), dtype=numba.float32)
    buf_idx = 0
    buf[buf_idx][i] = x[i]

    pass_num = 1
    while pass_num <= n:
      next_buf_idx = (buf_idx + 1) % 2
      if i - pass_num >= 0:
        buf[next_buf_idx][i] = buf[buf_idx][i] + buf[buf_idx][i - pass_num]
      cuda.syncthreads()

      # update variables for next iteration
      buf_idx = next_buf_idx
      pass_num *= 2

    out[i] = buf[buf_idx][i]

scan_kernel[blocks, threads](x, out)
res = out.copy_to_host()
print(res)





/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


[ 0.  1.  3.  6. 10. 14. 18. 22.]
